# UrbanPulse Adapter Continuation Training

This notebook continues fine-tuning your existing UrbanPulse LoRA adapter.

What you need to upload to Colab:
- `urbanpulse_multilang_qlora.jsonl`
- your adapter folder or zip with:
  - `adapter_model.safetensors`
  - `adapter_config.json`
  - tokenizer files


## 1. Runtime

In Colab, enable GPU:
- `Runtime` -> `Change runtime type`
- Hardware accelerator: `GPU`


In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
!pip install -U transformers peft accelerate datasets trl safetensors bitsandbytes

## 3. Upload dataset and adapter

Upload either:
- a zip archive of your adapter folder, or
- all adapter files directly


In [ ]:
from google.colab import files

uploaded = files.upload()

## 4. Prepare files

If you uploaded a zip, this cell will unpack it.
If you uploaded raw adapter files, it will move them into one folder.

In [ ]:
import os
import zipfile
from pathlib import Path
import shutil

BASE_DIR = Path('/content')
ADAPTER_DIR = BASE_DIR / 'urbanpulse_adapter'
OUTPUT_DIR = BASE_DIR / 'urbanpulse_adapter_continued'
DATASET_PATH = BASE_DIR / 'urbanpulse_multilang_qlora.jsonl'

ADAPTER_DIR.mkdir(exist_ok=True)

for item in BASE_DIR.iterdir():
    if item.suffix.lower() == '.zip' and 'urbanpulse' in item.name.lower():
        with zipfile.ZipFile(item, 'r') as archive:
            archive.extractall(ADAPTER_DIR)

raw_files = [
    'adapter_model.safetensors',
    'adapter_config.json',
    'tokenizer.json',
    'tokenizer_config.json',
    'chat_template.jinja',
    'README.md',
]

for filename in raw_files:
    src = BASE_DIR / filename
    dst = ADAPTER_DIR / filename
    if src.exists() and not dst.exists():
        shutil.move(str(src), str(dst))

print('Dataset exists:', DATASET_PATH.exists())
print('Adapter dir:', ADAPTER_DIR)
print('Adapter contents:', list(ADAPTER_DIR.rglob('*'))[:20])

## 5. Continue fine-tuning the adapter

In [ ]:
import torch
from datasets import load_dataset
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer
from trl import SFTTrainer, SFTConfig

MAX_SEQ_LENGTH = 1024
EPOCHS = 1.0
LEARNING_RATE = 1e-4

def format_sample(example):
    messages = example['messages']
    system_text = messages[0]['content']
    user_text = messages[1]['content']
    assistant_text = messages[2]['content']
    text = (
        f'<|system|>\n{system_text}\n'
        f'<|user|>\n{user_text}\n'
        f'<|assistant|>\n{assistant_text}'
    )
    return {'text': text}

model = AutoPeftModelForCausalLM.from_pretrained(
    str(ADAPTER_DIR),
    is_trainable=True,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
)
tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR), trust_remote_code=True)

dataset = load_dataset('json', data_files=str(DATASET_PATH), split='train')
dataset = dataset.map(format_sample, remove_columns=dataset.column_names)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    args=SFTConfig(
        output_dir=str(OUTPUT_DIR),
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        logging_steps=5,
        save_strategy='epoch',
        report_to='none',
        max_seq_length=MAX_SEQ_LENGTH,
        fp16=torch.cuda.is_available(),
        bf16=False,
    ),
)

trainer.train()
trainer.model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print('Saved continued adapter to:', OUTPUT_DIR)

## 6. Download the continued adapter

In [ ]:
!zip -r /content/urbanpulse_adapter_continued.zip /content/urbanpulse_adapter_continued

In [ ]:
from google.colab import files
files.download('/content/urbanpulse_adapter_continued.zip')